## But: Mise en place d'un RAG EMSI-ChatBot              
Prof. bousmah@gmail.com
## T.A.F: 
<ol>
<li>Démarrer le programme </li>
<li>Analyser le code </li>
<li>Améliorer le programme </li>
</ol>

In [3]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms import Ollama
from langchain.chains import RetrievalQA

# Data Source (PDF)

In [4]:
# Charger le document
loader = PyPDFLoader("emsi.pdf")
documents = loader.load()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)


# Chunking (ou diviser le texte en Chunk)

In [33]:
# Diviser le texte ou Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents) # texts est une liste d'objets de type Document (de langchain.schema)

Chaque Document contient deux informations importantes:

doc.page_content : le texte du chunk.

doc.metadata : les metadonnées du document (par exemple, numéro de page).

In [34]:
# Affichage des chunks
for i, doc in enumerate(texts):
    print(f"----- Chunk {i+1} -----")
    print(doc.page_content)
    print(f"--- Metadata: {doc.metadata} ---")
    print()


----- Chunk 1 -----
L'EMSI, ou École Marocaine des Sciences de l'Ingénieur, est un réseau d'écoles d'ingénieurs privées au Maroc, reconnu par l'État. Voici une description de l'EMSI : Histoire et Mission • Fondée en 1986, l'EMSI est l'une des premières écoles d'ingénieurs privées au Maroc. • Sa mission est de former des ingénieurs de haut niveau, opérationnels et compétents, capables de s'adapter aux évolutions technologiques et aux besoins du marché du travail. Campus et Implantation • L'EMSI dispose de plusieurs campus dans les principales villes du Maroc : o Casablanca (plusieurs campus) o Rabat o Marrakech o Tanger o Fès • Cette large implantation permet à l'EMSI d'être proche des bassins d'emploi et de contribuer au développement régional. Formations et Spécialités • Cycle d'ingénieur (Bac+5) : L'EMSI propose un cycle d'ingénieur en 5 ans, accessible après le baccalauréat (ﬁlière scientiﬁque) ou via des admissions parallèles (après une classe préparatoire, un BTS, un DUT, ou une l

# Créer les embeddings


In [35]:
# 3. Créer les embeddings
embeddings = HuggingFaceEmbeddings()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22228\94980471.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


In [36]:
# Stocker les embeddings
vectorstore = Chroma.from_documents(texts, embeddings)

In [37]:
# 5. Configurer Ollama
llm = Ollama(model="llama3.2:1b")  

In [38]:
# 6. Créer la chaîne QA
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())

In [39]:
qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(),
            return_source_documents=True,
        )

chain_type="stuff":


"stuff" est un type de chaîne qui indique que tous les documents récupérés seront combinés en un seul contexte qui sera ensuite transmis au LLM. En d'autres termes, tous les documents pertinents trouvés par la base de données vectorielle seront "entassés" dans le prompt du LLM.

Il existe d'autres types de chaines (comme map_reduce, refine, etc) qui traitent les documents récupérés d'une manière différente (par exemple en les traitant séquentiellement). Le choix du type de chaine est un compromis entre le temps de traitement, l'efficacité du prompt, et la taille du texte transmis. stuff est la manière la plus simple, et marche bien si la taille des documents et le nombre de documents retournés sont raisonnables.

return_source_documents=True

Sans ce paramètre (return_source_documents=False ou non spécifié): La chaîne renvoie uniquement la réponse générée par le LLM.

Avec ce paramètre (return_source_documents=True): La chaîne renvoie la réponse générée par le LLM, ainsi que la liste des documents sources qui ont contribué à la génération de la réponse.

In [40]:
# Exemple d'une question
query = "Histoire et Mission  de l'EMSI ?"
result = qa_chain(query)

# print(result)
print(result["result"])

Je sais que l'école d'ingénieurs EMSI (École supérieure d'ingénierie industrielle) est fondée en 1986 et que son histoire et sa mission sont connues.
